In [ ]:
"""

### 1. Step-by-Step Breakdown of the Process

#### **Step 1: Dataset Ingestion & Inspection**
- **Data Source**: Fetches the classic Box-Jenkins Airline Passengers dataset (`airline-passengers.csv`) directly from GitHub via `keras.utils.get_file()`.
- **Structure**: Contains monthly airline passenger counts from January 1949 to December 1960.
- **Initial Shape**: `(144, 1)` representing 144 continuous months (12 years).

#### **Step 2: Preprocessing & Sequence Framing (Sliding Window)**
- **Min-Max Scaling**: Normalizes passenger counts into the range $[0, 1]$ using `MinMaxScaler()`, which helps gradient-based neural network optimizers converge smoothly.
- **Look-back Window Framing (`LOOK_BACK = 12`)**:
  - Time series data is converted into supervised learning sequences $(X, y)$.
  - A sliding window of 12 past consecutive months ($t-11$ to $t$) is used as features ($X$) to predict the next month's passenger count ($y$ at $t+1$).
  - For 144 points, this produces $144 - 12 = 132$ sequence samples.

#### **Step 3: Chronological Train-Test Split & 3D Reshaping**
- **Split**: An $80/20$ chronological split is applied (no shuffling to prevent lookahead data leakage).
  - **Train set**: First 80% (105 samples).
  - **Test set**: Remaining 20% (27 samples).
- **Reshaping**: Recurrent layers in Keras require 3D input tensors: `[samples, timesteps, features]`.
  - `X_train` is reshaped to `(105, 12, 1)`.
  - `X_test` is reshaped to `(27, 12, 1)`.

#### **Step 4: Model Architectures & Training**
Three separate models are constructed with identical layer setups for a fair benchmark:
1. **Model A (SimpleRNN)**: `SimpleRNN(50, activation="tanh")` $\rightarrow$ `Dense(1)`
2. **Model B (LSTM)**: `LSTM(50, activation="tanh")` $\rightarrow$ `Dense(1)`
3. **Model C (GRU)**: `GRU(50, activation="tanh")` $\rightarrow$ `Dense(1)`

- **Training Hyperparameters**:
  - **Optimizer**: `Adam`
  - **Loss Function**: Mean Squared Error (`mse`)
  - **Epochs**: `100`
  - **Batch Size**: `8`
  - **Validation**: Evaluated against test data `(X_test, y_test)` during training.

#### **Step 5: Post-Processing & Evaluation**
- Predictions for both train and test sets are transformed back to actual passenger numbers using `scaler.inverse_transform()`.
- **Metric**: Root Mean Squared Error (**RMSE**) in terms of thousands of passengers.

---

### 2. Existing Outputs from the Notebook

Based on the execution cells saved in the notebook, here are the outputs:

#### **A. Dataset and Sequence Shapes**
```text
Dataset shape: (144, 1)
     Month  Passengers
0  1949-01         112
1  1949-02         118
2  1949-03         132
3  1949-04         129
4  1949-05         121

Train shape: (105, 12, 1) Test shape: (27, 12, 1)
```


#### **C. Generated Visualizations / Artifacts**
The code saves 5 plots to disk:
1. `raw_series.png`: Visualization of the raw passenger trajectory over the entire 12-year window (showing upward trend and yearly seasonality).
2. `simplernn_loss.png`: Training loss vs. validation loss curve across 100 epochs for SimpleRNN.
3. `lstm_loss.png`: Loss curves across 100 epochs for LSTM.
4. `gru_loss.png`: Loss curves across 100 epochs for GRU.
5. `rnn_vs_lstm_forecast.png`: Overlaid test-set predictions comparing Actual values against SimpleRNN, LSTM, and GRU predictions.

---

### 3. Key Observations & Analysis

- **Why did SimpleRNN outperform LSTM and GRU on this dataset?**
  - **Dataset size constraint**: The dataset only has 105 training samples.
  - **Parameter Complexity**: LSTM has 4 gates/states and GRU has 2 gates, meaning they have significantly more parameters than SimpleRNN. With very limited training sequences, LSTM overfits the training set and generalizes less effectively on the test set (Test RMSE 61.83 vs 34.86).
  - SimpleRNN's lower parameter count acts as natural regularization for small, seasonal datasets.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.utils import get_file
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, LSTM, GRU, Dense
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

# -------------------------------------------------------------------
# 1. Load the dataset
# -------------------------------------------------------------------
# keras.utils.get_file downloads and caches the file (~/.keras/datasets/),
# so this only happens once.
DATA_URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
csv_path = get_file("airline-passengers.csv", origin=DATA_URL)

df = pd.read_csv(csv_path)
# The CSV has columns: Month, Passengers
series = df.iloc[:, 1].values.astype("float32").reshape(-1, 1)

print("Dataset shape:", series.shape)
print(df.head())

plt.figure(figsize=(10, 4))
plt.plot(series)
plt.title("International Airline Passengers (1949-1960)")
plt.xlabel("Month")
plt.ylabel("Passengers (thousands)")
plt.tight_layout()
plt.savefig("raw_series.png")
plt.close()

# -------------------------------------------------------------------
# 2. Preprocess: scale to [0, 1] and build sliding-window sequences
# -------------------------------------------------------------------
scaler = MinMaxScaler(feature_range=(0, 1))
scaled = scaler.fit_transform(series)

LOOK_BACK = 12  # use the previous 12 months to predict the next month


def create_dataset(data, look_back=1):
    X, y = [], []
    for i in range(len(data) - look_back):
        X.append(data[i:i + look_back, 0])
        y.append(data[i + look_back, 0])
    return np.array(X), np.array(y)


X, y = create_dataset(scaled, LOOK_BACK)

# Train/test split (chronological, no shuffling for time series)
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Reshape to [samples, timesteps, features] as required by RNN/LSTM layers
X_train = X_train.reshape((X_train.shape[0], LOOK_BACK, 1))
X_test = X_test.reshape((X_test.shape[0], LOOK_BACK, 1))

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

# -------------------------------------------------------------------
# 3. Helper to train a model and report results
# -------------------------------------------------------------------


def train_and_evaluate(model, name, epochs=100):
    model.compile(optimizer="adam", loss="mse")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=epochs, batch_size=8, verbose=0
    )

    train_pred = model.predict(X_train, verbose=0)
    test_pred = model.predict(X_test, verbose=0)

    # Invert scaling to get back to real passenger numbers
    train_pred_inv = scaler.inverse_transform(train_pred)
    y_train_inv = scaler.inverse_transform(y_train.reshape(-1, 1))
    test_pred_inv = scaler.inverse_transform(test_pred)
    y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1))

    train_rmse = np.sqrt(mean_squared_error(y_train_inv, train_pred_inv))
    test_rmse = np.sqrt(mean_squared_error(y_test_inv, test_pred_inv))
    print(f"\n[{name}] Train RMSE: {train_rmse:.2f} | Test RMSE: {test_rmse:.2f}")

    plt.figure(figsize=(6, 4))
    plt.plot(history.history["loss"], label="train loss")
    plt.plot(history.history["val_loss"], label="val loss")
    plt.title(f"{name} - Loss curve")
    plt.xlabel("Epoch")
    plt.ylabel("MSE")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{name.lower()}_loss.png")
    plt.close()

    return test_pred_inv, y_test_inv, test_rmse


# -------------------------------------------------------------------
# 4. Model A: Simple RNN
# -------------------------------------------------------------------
rnn_model = Sequential([
    SimpleRNN(50, activation="tanh", input_shape=(LOOK_BACK, 1)),
    Dense(1)
])
rnn_pred, y_test_inv, rnn_rmse = train_and_evaluate(rnn_model, "SimpleRNN")

# -------------------------------------------------------------------
# 5. Model B: LSTM
# -------------------------------------------------------------------
lstm_model = Sequential([
    LSTM(50, activation="tanh", input_shape=(LOOK_BACK, 1)),
    Dense(1)
])
lstm_pred, _, lstm_rmse = train_and_evaluate(lstm_model, "LSTM")

# -------------------------------------------------------------------
# 6. Model C: GRU (Gated Recurrent Unit)
# -------------------------------------------------------------------
gru_model = Sequential([
    GRU(50, activation="tanh", input_shape=(LOOK_BACK, 1)),
    Dense(1)
])
gru_pred, _, gru_rmse = train_and_evaluate(gru_model, "GRU")

# -------------------------------------------------------------------
# 7. Compare predictions on the test set
# -------------------------------------------------------------------
plt.figure(figsize=(10, 5))
plt.plot(y_test_inv, label="Actual", linewidth=2)
plt.plot(rnn_pred, label=f"SimpleRNN (RMSE={rnn_rmse:.1f})")
plt.plot(lstm_pred, label=f"LSTM (RMSE={lstm_rmse:.1f})")
plt.plot(gru_pred, label=f"GRU (RMSE={gru_rmse:.1f})")
plt.title("Airline Passengers Forecast: RNN vs LSTM vs GRU")
plt.xlabel("Time step (test set)")
plt.ylabel("Passengers (thousands)")
plt.legend()
plt.tight_layout()
plt.savefig("rnn_vs_lstm_forecast.png")
plt.close()

print("\nSaved plots: raw_series.png, simplernn_loss.png, lstm_loss.png, "
      "gru_loss.png, rnn_vs_lstm_forecast.png")
print(f"\nSummary -> SimpleRNN: {rnn_rmse:.2f} | LSTM: {lstm_rmse:.2f} | GRU: {gru_rmse:.2f}")

Dataset shape: (144, 1)
     Month  Passengers
0  1949-01         112
1  1949-02         118
2  1949-03         132
3  1949-04         129
4  1949-05         121
Train shape: (105, 12, 1) Test shape: (27, 12, 1)


c:\Users\sunit\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



[SimpleRNN] Train RMSE: 12.32 | Test RMSE: 34.86


c:\Users\sunit\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



[LSTM] Train RMSE: 19.33 | Test RMSE: 61.83


c:\Users\sunit\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



[GRU] Train RMSE: 20.97 | Test RMSE: 44.32

Saved plots: raw_series.png, simplernn_loss.png, lstm_loss.png, gru_loss.png, rnn_vs_lstm_forecast.png

Summary -> SimpleRNN: 34.86 | LSTM: 61.83 | GRU: 44.32
